In [2]:
import pandas as pd

# Load and clean both sheets
file_path = "Data for technical assessment.xlsx"
sheet1_raw = pd.read_excel(file_path, sheet_name="Dataset 1 - General", engine="openpyxl", header=None)
sheet2_raw = pd.read_excel(file_path, sheet_name="Dataset 2 - Underwriting", engine="openpyxl", header=None)

# Merge first two rows into header
header1 = [f"{a} {b}".strip() for a, b in zip(sheet1_raw.iloc[0], sheet1_raw.iloc[1])]
sheet1 = sheet1_raw.iloc[2:].copy()
sheet1.columns = header1
sheet1.rename(columns={sheet1.columns[0]: "Firm"}, inplace=True)

header2 = [f"{a} {b}".strip() for a, b in zip(sheet2_raw.iloc[0], sheet2_raw.iloc[1])]
sheet2 = sheet2_raw.iloc[2:].copy()
sheet2.columns = header2
sheet2.rename(columns={sheet2.columns[0]: "Firm"}, inplace=True)

# Merge sheets
merged = pd.merge(sheet1, sheet2, on="Firm", how="inner")

In [3]:
merged

,Firm,NWP (£m) 2016YE,NWP (£m) 2017YE,NWP (£m) 2018YE,NWP (£m) 2019YE,NWP (£m) 2020YE,SCR (£m) 2016YE,SCR (£m) 2017YE,SCR (£m) 2018YE,SCR (£m) 2019YE,...,Gross expense ratio 2016YE,Gross expense ratio 2017YE,Gross expense ratio 2018YE,Gross expense ratio 2019YE,Gross expense ratio 2020YE,Gross combined ratio 2016YE,Gross combined ratio 2017YE,Gross combined ratio 2018YE,Gross combined ratio 2019YE,Gross combined ratio 2020YE
0,Firm 1,-13779.815629,0,0,0,0,1085.360139,0.0,0,0,...,0,56.813725,0,0,0,0,68.215239,0,0,0
1,Firm 2,28.178059,26.865049,25.064438,23.226445,21.718558,10.190314,10.113572,9.495235,8.146471,...,0.743265,0.963451,0.814588,0,0,0.945394,1.126744,0.939197,0,0
2,Firm 3,0,75.609681,70.578732,78.432782,85.73583,322.955115,363.782327,362.290859,394.295982,...,0,0,0,0,0,0,0,0,0,0
3,Firm 4,22344.199923,23963.910709,25760.390158,25512.748836,24996.021042,16573.6448,16332.7488,17103.616,17219.24608,...,0.14393,0.147519,0.092971,0.054781,-0.546237,0.848032,1.474778,1.727968,1.208823,-10.736084
4,Firm 5,68.200993,51.663132,44.010833,42.008556,81.273653,52.824396,38.053768,34.696815,57.231788,...,0.177212,0.13431,0.109074,0.121044,0.109187,0.508711,1.259454,1.304168,0.983277,0.997184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,Firm 321,0,0,-1.011367,-6.599067,24.632234,0,0.258621,62.227588,51.830942,...,0.211938,0.256118,0.245704,0.236224,0.278674,0.978004,1.002691,0.97254,0.958443,0.81687
321,Firm 322,2092.156137,2084.124818,2022.212247,2103.048716,2029.697013,1711.220667,1641.309461,1329.471064,1399.098954,...,0.364543,0.372169,0.39877,0.420327,0.373813,0.885956,0.960993,0.913687,0.943246,0.995833
322,Firm 323,0,0,0,0,0,30.438558,15.232621,5.332069,1.55137,...,0,0,0,0,0,0,0,0,0,0
323,Firm 324,23.41538,22.650321,24.268465,25.811984,26.546638,32.096633,30.205948,29.517977,29.954935,...,0.427635,0.371681,0.357627,0.330893,0.302577,1.063136,1.006945,0.982816,0.994712,0.780065


In [25]:
# in merged data, for each year, check if NWP <= GWP for each firm
years = [col.split()[-1] for col in merged.columns if "GWP" in col]

errors = []
for year in years:
    gwp_col = f"GWP (£m) {year}"
    nwp_col = f"NWP (£m) {year}"
    for idx, row in merged.iterrows():
        print(row)
        if row[nwp_col] > row[gwp_col]:
            errors.append({
                "Firm": row["Firm"],
                "Year": year,
                "GWP": row[gwp_col],
                "NWP": row[nwp_col]
            })
errors_df = pd.DataFrame(errors)
errors_df

Firm                                 Firm 1
NWP (£m)  2016YE              -13779.815629
NWP (£m)  2017YE                          0
NWP (£m)  2018YE                          0
NWP (£m)  2019YE                          0
                                   ...     
Gross combined ratio 2016YE               0
Gross combined ratio 2017YE       68.215239
Gross combined ratio 2018YE               0
Gross combined ratio 2019YE               0
Gross combined ratio 2020YE               0
Name: 0, Length: 86, dtype: object


KeyError: 'NWP (£m) 2016YE'

In [23]:
years

['2016YE', '2017YE', '2018YE', '2019YE', '2020YE']

In [28]:
# Extract key metrics
metrics = ["GWP (£m)", "NWP (£m)", "SCR coverage ratio", "Gross claims incurred (£m)", "Net combined ratio"]
metric_cols = [col for col in merged.columns if any(m in col for m in metrics)]

# Tidy dataframe
records = []
for _, row in merged.iterrows():
    firm = row['Firm']
    for col in metric_cols:
        parts = col.split()
        metric = " ".join(parts[:-1])
        year = parts[-1]
        try:
            value = float(row[col])
        except:
            value = None
        records.append({"Firm": firm, "Year": year, "Metric": metric, "Value": value})

tidy_df = pd.DataFrame(records)

In [20]:
#some data sense checks
# for each year, check if NWP <= GWP for each firm


In [29]:
tidy_df

,Firm,Year,Metric,Value
0,Firm 1,2016YE,NWP (£m),-13779.815629
1,Firm 1,2017YE,NWP (£m),0.000000
2,Firm 1,2018YE,NWP (£m),0.000000
3,Firm 1,2019YE,NWP (£m),0.000000
4,Firm 1,2020YE,NWP (£m),0.000000
...,...,...,...,...
8120,Firm 325,2016YE,Net combined ratio,0.000000
8121,Firm 325,2017YE,Net combined ratio,0.000000
8122,Firm 325,2018YE,Net combined ratio,0.000000
8123,Firm 325,2019YE,Net combined ratio,0.000000


In [4]:
# Summary for prioritization
summary = tidy_df.groupby(['Firm', 'Metric']).agg({"Value": ["mean", "std"]}).reset_index()
summary.columns = ['Firm', 'Metric', 'AvgValue', 'StdDev']

# Identify top firms and outliers
largest_firms = summary[summary['Metric'] == 'GWP (£m)'].sort_values('AvgValue', ascending=False).head(10)
volatile_firms = summary.sort_values('StdDev', ascending=False).head(10)
outliers_scr = summary[(summary['Metric'] == 'SCR coverage ratio') & (summary['AvgValue'] < 1)].sort_values('AvgValue').head(10)
outliers_combined = summary[(summary['Metric'] == 'Net combined ratio') & (summary['AvgValue'] > 1)].sort_values('AvgValue', ascending=False).head(10)

In [17]:
summary

,Firm,Metric,AvgValue,StdDev
0,Firm 1,GWP (£m),2.818970e+02,6.303408e+02
1,Firm 1,Gross claims incurred (£m),9.334784e-03,2.087321e-02
2,Firm 1,NWP (£m),-2.755963e+03,6.162521e+03
3,Firm 1,Net combined ratio,1.364305e+01,3.050678e+01
4,Firm 1,SCR coverage ratio,9.635840e+07,2.154639e+08
...,...,...,...,...
1620,Firm 99,GWP (£m),4.535788e+02,1.457792e+01
1621,Firm 99,Gross claims incurred (£m),2.115106e+01,2.659385e+01
1622,Firm 99,NWP (£m),1.676096e+02,6.688409e+00
1623,Firm 99,Net combined ratio,-9.223339e+03,2.062401e+04


In [12]:
# Detect reporting errors: extreme year-on-year changes (>300% change)
error_records = []
for firm in tidy_df['Firm'].unique():
    firm_data = tidy_df[tidy_df['Firm'] == firm]
    for metric in metrics:
        metric_data = firm_data[firm_data['Metric'] == metric].sort_values('Year')
        values = metric_data['Value'].tolist()
        for i in range(1, len(values)):
            if values[i-1] and values[i] and abs(values[i] - values[i-1]) > 3 * abs(values[i-1]):
                error_records.append({"Firm": firm, "Metric": metric, "PrevValue": values[i-1], "CurrentValue": values[i]})
errors_df = pd.DataFrame(error_records)

In [16]:
errors_df['year-on-year change'] = ((errors_df['CurrentValue'] - errors_df['PrevValue']) / errors_df['PrevValue']).abs() * 100

#print the top 20 errors detected
errors_df.sort_values('year-on-year change', ascending=False).head(20)

,Firm,Metric,PrevValue,CurrentValue,year-on-year change
119,Firm 216,SCR coverage ratio,1.493878,9.635840e+08,6.450221e+10
81,Firm 131,SCR coverage ratio,1.748408,4.817920e+08,2.755604e+10
0,Firm 1,SCR coverage ratio,1.979865,4.817920e+08,2.433459e+10
136,Firm 276,NWP (£m),-0.003253,5.714939e+03,1.756790e+08
47,Firm 66,SCR coverage ratio,5.203180,3.856018e+06,7.410878e+07
135,Firm 276,GWP (£m),0.010037,5.714942e+03,5.694040e+07
122,Firm 228,Gross claims incurred (£m),-0.000013,7.493312e-01,5.753008e+06
15,Firm 29,Gross claims incurred (£m),0.002773,1.425000e+01,5.137297e+05
143,Firm 306,Gross claims incurred (£m),0.013954,3.321084e+01,2.378981e+05
156,Firm 316,NWP (£m),-0.147332,-1.930833e+02,1.309528e+05


In [6]:
import plotly.express as px

# Create charts
fig_gwp = px.bar(largest_firms, x='Firm', y='AvgValue', title='Top 10 Firms by GWP (£m)')
# fig_gwp.write_image('largest_firms.png')
# fig_gwp.write_json('largest_firms.json')

fig_vol = px.bar(volatile_firms, x='Firm', y='StdDev', title='Top 10 Most Volatile Firms')
# fig_vol.write_image('volatile_firms.png')
# fig_vol.write_json('volatile_firms.json')

fig_scr = px.bar(outliers_scr, x='Firm', y='AvgValue', title='Low SCR Coverage Ratio (<1)')
# fig_scr.write_image('low_scr.png')
# fig_scr.write_json('low_scr.json')

fig_combined = px.bar(outliers_combined, x='Firm', y='AvgValue', title='High Net Combined Ratio (>1)')
# fig_combined.write_image('high_combined.png')
# fig_combined.write_json('high_combined.json')


In [7]:
fig_gwp.show()
fig_vol.show()
fig_scr.show()
fig_combined.show()

In [30]:
df_combined = tidy_df[tidy_df["Metric"] == "Net combined ratio"]
df_nwp = tidy_df[tidy_df["Metric"] == "NWP (£m)"]
df_gwp = tidy_df[tidy_df["Metric"] == "GWP (£m)"]

df_plot = pd.merge(df_combined, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
df_plot = pd.merge(df_plot, df_gwp, on=["Firm", "Year"], suffixes=("", "_gwp"))

fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)


In [31]:
fig_combined_vs_nwp.show()

In [10]:
# plotting the same as above, restricting net combined ratios up to 1000 and >= 0

df_combined_restricted = tidy_df[
    (tidy_df["Metric"] == "Net combined ratio") &
    (tidy_df["Value"] <= 1000) &
    (tidy_df["Value"] >= -1000)
]

df_nwp = tidy_df[tidy_df["Metric"] == "NWP (£m)"]

df_plot = pd.merge(df_combined_restricted, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))

fig_combined_vs_nwp_restricted = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)

In [11]:
fig_combined_vs_nwp_restricted.show()

In [38]:
#plot NWP vs GWP for 2020
fig_gwp_vs_nwp = px.scatter(
    df_plot,#[df_plot["Year"] == '2020YE'],
    x="Value",
    y="Value_nwp",
    title="Gross Written Premium vs Net Written Premium for 2020",
    labels={"Value": "Gross Written Premium (£m)", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm"]
)


In [39]:
fig_gwp_vs_nwp.show()